# 06 · Yeni ayı donmuş modellerle uygulama ve dashboard

Müşteri giriş/çıkış ve devirleri yeni ayın hazırlanmış verisinde bulunmalıdır. Bu adım
değişken seçimini veya kümeleri yeniden eğitmez. Donmuş şema ve aynı ana yöntem uygulanır;
eksik alan, yeni coğrafi grup veya kapsam/boyut sorunu varsa çekirdek kod işlemi durdurur.
Güncel ana model yoksa bu notebook da yeni atama üretmez.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


In [ ]:
import pandas as pd
YENI_AYI_UYGULA = False
YENI_AY = "2027-01"
YENI_VERI_DOSYASI = ROOT / "veri/yeni_ay.csv"  # Kendi dosyanın yolunu yaz
selection = qualified_selection()
print("Onaylı model:", selection)
display(current_decision())


In [ ]:
if YENI_AYI_UYGULA:
    if selection is None:
        print("Atama yapılmadı: önce 04’te bütün koşulları sağlayan ortak bir model gerekli.")
    else:
        require_files(YENI_VERI_DOSYASI)
        from dinamik_secim.runner import apply_schema
        from kumeleme_secimi.runner import infer
        prepared = apply_schema(selection / "selected_schema.json", YENI_VERI_DOSYASI, fresh_run("aylik_hazirlik"), as_of=YENI_AY)
        assignments = infer(selection, prepared / "cluster_input_latest.csv", period=YENI_AY)
        target = fresh_run("aylik_atamalar")
        target.mkdir()
        assignments.to_csv(target / "sube_atamalari.csv", index=False, encoding="utf-8-sig")
        display(assignments.head())
        display(read_json(prepared / "drift.json"))
        print("Yeni ay atamaları:", target)
else:
    print("Yeni veri okunmadı ve atama dosyası üretilmedi.")


## Dashboard ve çalışma klasörleri

Jupyter denemeleri `outputs/jupyter/` altındadır; tarihsel ana sonuçları veya dashboard’u
kendiliğinden değiştirmez. Yeni bir denemeyi ana çalışma olarak aktive etme bu notebook’ların
işi değildir. Mevcut dashboard hazır anlık görüntüdür.

Tarayıcıda açmak için proje kökünde terminalden:

```bash
python -m http.server 8765 --bind 127.0.0.1 --directory model_dashboard/dist
```

Sonra <http://127.0.0.1:8765/> adresini aç. JupyterLab HTML önizlemesi JavaScript’i
çalıştırmayabilir; normal tarayıcı kullan.


In [ ]:
rows = []
for stage in ["dinamik_feature_selection", "kumeleme_karsilastirma", "otomatik_kumeleme", "dea_verimlilik", "dea_girdi_kontrol"]:
    run = latest_run(stage)
    rows.append({"aşama": stage, "yerel_koşu": str(run) if run else "hazır dashboard tabloları", "not": "Jupyter sonucu önceliklidir; otomatik ana aktivasyon yapılmaz"})
display(pd.DataFrame(rows))


## İsteğe bağlı: ana çalışma çıktılarından dashboard’u yeniden oluştur

Bu komut `outputs/` altındaki **ana** koşuları okur; Jupyter denemelerini aktive etmez.
Tam tarihsel kaynaklar Git klonunda bulunmayabilir. Yalnızca kaynaklar mevcutsa açın;
salt notebook kullanımı için bu hücreyi çalıştırmak gerekmez.


In [ ]:
DASHBOARDU_YENIDEN_HAZIRLA = False
if DASHBOARDU_YENIDEN_HAZIRLA:
    required = [ROOT / "outputs/lightgbm_ilk_model/sonuclar/rapor_verisi.json", ROOT / "outputs/banka_test_verisi_v3/manifest.json", ROOT / "outputs/banka_test_verisi_v3/ocak2027_banka_hedefleri.csv"]
    require_files(*required)
    run_python("dashboard_veri_hazirla.py")
else:
    print("Hazır dashboard korunuyor:", ROOT / "model_dashboard/dist/index.html")
